<a href="https://colab.research.google.com/github/suri307/MLA0205-FOML/blob/main/exp3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import csv
import math

a = []

with open('/content/play_tennis.csv', 'r') as csvfile:
    for row in csv.reader(csvfile):
        a.append(row)

print(a)

print("\n The total number of training instances are : ", len(a))

attributes = a[0][:-1]
data = a[1:]

def entropy(data):
    total = len(data)

    yes = sum(1 for row in data if row[-1] == 'yes')
    no = sum(1 for row in data if row[-1] == 'no')

    if yes == 0 or no == 0:
        return 0

    p_yes = yes / total
    p_no = no / total

    return -(p_yes * math.log2(p_yes) +
             p_no * math.log2(p_no))

def information_gain(data, index):
    total_entropy = entropy(data)

    values = set(row[index] for row in data)

    weighted_entropy = 0

    for value in values:
        subset = [row for row in data if row[index] == value]

        weighted_entropy += (len(subset) / len(data)) * entropy(subset)

    return total_entropy - weighted_entropy

def id3(data, attributes, attribute_names):

    classes = [row[-1] for row in data]

    if classes.count(classes[0]) == len(classes):
        return classes[0]

    if len(attributes) == 0:
        return max(set(classes), key=classes.count)

    gains = []

    for index in attributes:
        gains.append(information_gain(data, index))

    best_index = attributes[gains.index(max(gains))]
    best_attribute = attribute_names[attributes.index(best_index)]

    tree = {best_attribute: {}}

    values = set(row[best_index] for row in data)

    for value in values:

        subset = [row for row in data if row[best_index] == value]

        remaining_attributes = [
            index for index in attributes
            if index != best_index
        ]

        remaining_names = [
            name for index, name in zip(attributes, attribute_names)
            if index != best_index
        ]

        tree[best_attribute][value] = id3(
            subset,
            remaining_attributes,
            remaining_names
        )

    return tree

attribute_indices = list(range(len(attributes)))

tree = id3(data, attribute_indices, attributes)

print("\n The Decision Tree is : ")
print(tree)

def classify(tree, sample):

    if not isinstance(tree, dict):
        return tree

    attribute = list(tree.keys())[0]

    value = sample[attribute]

    if value in tree[attribute]:
        return classify(tree[attribute][value], sample)
    else:
        return "Unknown"

new_sample = {
    'Outlook': 'sunny',
    'Temperature': 'cool',
    'Humidity': 'high',
    'Wind': 'strong'
}

print("\n The new sample is : ")
print(new_sample)

result = classify(tree, new_sample)

print("\n The classification result is : ", result)

[['Outlook', 'Temperature', 'Humidity', 'Wind', 'Target'], ['sunny', 'hot', 'high', 'weak', 'no'], ['sunny', 'hot', 'high', 'strong', 'no'], ['overcast', 'hot', 'high', 'weak', 'yes'], ['rain', 'mild', 'high', 'weak', 'yes'], ['rain', 'cool', 'normal', 'weak', 'yes'], ['rain', 'cool', 'normal', 'strong', 'no'], ['overcast', 'cool', 'normal', 'strong', 'yes'], ['sunny', 'mild', 'high', 'weak', 'no'], ['sunny', 'cool', 'normal', 'weak', 'yes'], ['rain', 'mild', 'normal', 'weak', 'yes'], ['sunny', 'mild', 'normal', 'strong', 'yes'], ['overcast', 'mild', 'high', 'strong', 'yes'], ['overcast', 'hot', 'normal', 'weak', 'yes'], ['rain', 'mild', 'high', 'strong', 'no']]

 The total number of training instances are :  15

 The Decision Tree is : 
{'Outlook': {'rain': {'Wind': {'strong': 'no', 'weak': 'yes'}}, 'overcast': 'yes', 'sunny': {'Humidity': {'normal': 'yes', 'high': 'no'}}}}

 The new sample is : 
{'Outlook': 'sunny', 'Temperature': 'cool', 'Humidity': 'high', 'Wind': 'strong'}

 The c